# 01. Basic Agent - Microsoft Agent Framework Edition

## 개요
이 노트북에서는 Microsoft Agent Framework를 사용하여 기본 ChatAgent를 구현합니다.

### 학습 목표
- ChatAgent의 기본 구조 이해
- Azure OpenAI와의 통합
- 도구(Tools) 통합
- 스트리밍 응답 처리
- 대화 상태 관리

## 1. 환경 설정

In [1]:
import os
import asyncio
from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()

# Azure OpenAI 설정 확인
print(f"Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Deployment: {os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")
print(f"API Key exists: {bool(os.getenv('AZURE_OPENAI_API_KEY'))}")

Endpoint: https://eusonopenai.openai.azure.com/
Deployment: gpt-4o-mini
API Key exists: True


## 2. 기본 ChatAgent 생성

Microsoft Agent Framework의 ChatAgent는 대화형 AI 에이전트의 기본 구성 요소입니다.

In [2]:
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import DefaultAzureCredential

# Azure OpenAI Chat Client 생성
# API Key 기반 인증 사용
chat_client = AzureOpenAIChatClient(
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
)

# ChatAgent 생성
agent = ChatAgent(
    chat_client=chat_client,
    instructions="You are a helpful AI assistant. Answer questions clearly and concisely.",
    name="Assistant"
)

print("✅ ChatAgent 생성 완료")

✅ ChatAgent 생성 완료


## 3. 기본 대화 실행

In [3]:
# 기본 대화
async def basic_chat():
    result = await agent.run("What is artificial intelligence?")
    print(f"🤖 Assistant: {result.text}")

await basic_chat()

🤖 Assistant: Artificial intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. These processes include learning (the acquisition of information and rules for using it), reasoning (using rules to reach approximate or definite conclusions), and self-correction. AI can be categorized into two main types: 

1. **Narrow AI**: Designed to perform a specific task, such as voice recognition, image analysis, or playing games. It operates under a limited set of constraints and parameters.

2. **General AI**: A theoretical form of AI that would have the ability to understand and reason about the world as a human would, performing any intellectual task that a human can do.

AI technologies include machine learning, natural language processing, robotics, and more, enabling applications across various fields such as healthcare, finance, transportation, and entertainment.


## 4. 도구(Tools)를 가진 에이전트

에이전트에 외부 도구를 통합하여 기능을 확장할 수 있습니다.

In [4]:
from typing import Annotated
from pydantic import Field
from random import randint

# 날씨 조회 도구
def get_weather(
    location: Annotated[str, Field(description="The location to get weather for")]
) -> str:
    """Get the current weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    temp = randint(10, 30)
    condition = conditions[randint(0, 3)]
    return f"The weather in {location} is {condition} with a temperature of {temp}°C."

# 환율 조회 도구
def get_exchange_rate(
    from_currency: Annotated[str, Field(description="Source currency code (e.g., USD)")],
    to_currency: Annotated[str, Field(description="Target currency code (e.g., KRW)")]
) -> str:
    """Get the exchange rate between two currencies."""
    # 실제로는 API를 호출하지만, 여기서는 예시 값 반환
    rates = {
        ("USD", "KRW"): 1320.50,
        ("USD", "EUR"): 0.92,
        ("EUR", "KRW"): 1435.80,
    }
    rate = rates.get((from_currency, to_currency), 1.0)
    return f"1 {from_currency} = {rate} {to_currency}"

# 도구가 있는 에이전트 생성
agent_with_tools = ChatAgent(
    chat_client=chat_client,
    instructions="You are a helpful assistant with access to weather and exchange rate information.",
    tools=[get_weather, get_exchange_rate],
    name="ToolAssistant"
)

print("✅ 도구가 통합된 ChatAgent 생성 완료")

✅ 도구가 통합된 ChatAgent 생성 완료


In [5]:
# 도구를 사용하는 대화
async def tool_chat():
    questions = [
        "What's the weather in Seoul?",
        "What's the exchange rate from USD to KRW?",
        "Tell me about both the weather in Tokyo and the USD to EUR exchange rate."
    ]
    
    for question in questions:
        print(f"\n👤 User: {question}")
        result = await agent_with_tools.run(question)
        print(f"🤖 Assistant: {result.text}")

await tool_chat()


👤 User: What's the weather in Seoul?
🤖 Assistant: The weather in Seoul is currently stormy with a temperature of 27°C.

👤 User: What's the exchange rate from USD to KRW?
🤖 Assistant: The exchange rate from USD to KRW is 1 USD = 1320.5 KRW.

👤 User: Tell me about both the weather in Tokyo and the USD to EUR exchange rate.
🤖 Assistant: The current weather in Tokyo is sunny with a temperature of 28°C. As for the exchange rate, 1 USD is equivalent to 0.92 EUR.


## 5. 스트리밍 응답

실시간으로 응답을 받아 사용자 경험을 향상시킬 수 있습니다.

In [6]:
async def streaming_chat():
    print("👤 User: Tell me a short story about a robot.")
    print("🤖 Assistant: ", end="", flush=True)
    
    async for chunk in agent.run_stream("Tell me a short story about a robot."):
        print(chunk.text, end="", flush=True)
    
    print("\n")

await streaming_chat()

👤 User: Tell me a short story about a robot.
🤖 Assistant: In a bustling future city filled with skyscrapers and flying cars, there lived a small, unassuming robot named Zeke. Zeke was designed for one purpose: to clean the streets of litter and debris. Every day, he patrolled a neighborhood, whirring softly as he collected trash, diligently following his programming.

One day, while picking up discarded items, Zeke noticed a small child crying on a park bench. Curious, he approached. The child, named Mia, was upset because she had lost her favorite toy, a bright red ball. Zeke's sensors could detect her distress, and he decided to help.

Using his built-in scanners, Zeke searched the park, analyzing every nook and cranny where the ball might have rolled. After some time, he located it trapped behind a bush. With a gentle whir, he retrieved the ball and rolled it back to Mia.

Her face lit up with joy as she saw her beloved toy. “Thank you, robot!” she exclaimed, hugging the ball tightl

## 6. 대화 히스토리 관리

Agent Framework는 AgentThread를 통해 대화 상태를 관리합니다.

In [7]:
from agent_framework import AgentThread

# 대화 스레드 생성
thread = AgentThread()

# 메모리를 가진 에이전트 생성
memory_agent = ChatAgent(
    chat_client=chat_client,
    instructions="You are a helpful assistant. Remember previous conversations.",
    name="MemoryAssistant"
)

async def memory_chat():
    # 첫 번째 대화
    print("👤 User: My name is John and I like pizza.")
    result1 = await memory_agent.run(
        "My name is John and I like pizza.",
        thread=thread
    )
    print(f"🤖 Assistant: {result1.text}\n")
    
    # 두 번째 대화 - 이전 대화를 기억하는지 확인
    print("👤 User: What's my name and what food do I like?")
    result2 = await memory_agent.run(
        "What's my name and what food do I like?",
        thread=thread
    )
    print(f"🤖 Assistant: {result2.text}")

await memory_chat()

👤 User: My name is John and I like pizza.


🤖 Assistant: Nice to meet you, John! Pizza is a delicious choice. What's your favorite type or topping?

👤 User: What's my name and what food do I like?
🤖 Assistant: Your name is John, and you like pizza!


## 7. 구조화된 출력

Pydantic 모델을 사용하여 구조화된 응답을 받을 수 있습니다.

In [8]:
from pydantic import BaseModel
from typing import List
import json

# 응답 스키마 정의
class MovieRecommendation(BaseModel):
    title: str
    genre: str
    year: int
    rating: float
    reason: str

class MovieList(BaseModel):
    recommendations: List[MovieRecommendation]

# 구조화된 출력을 사용하는 에이전트
# Agent Framework에서 구조화된 출력을 원할 경우, 프롬프트에 JSON 포맷 명시
structured_agent = ChatAgent(
    chat_client=chat_client,
    instructions="""You are a movie recommendation assistant. Provide structured recommendations.
    
IMPORTANT: You must respond with ONLY valid JSON in this exact format:
{
  "recommendations": [
    {
      "title": "Movie Title",
      "genre": "Genre",
      "year": 2024,
      "rating": 8.5,
      "reason": "Why this movie"
    }
  ]
}

Do not include any text before or after the JSON.""",
    name="MovieAssistant"
)

async def structured_output():
    result = await structured_agent.run(
        "Recommend 3 sci-fi movies from the last 5 years. Return ONLY valid JSON."
    )
    
    try:
        # 응답에서 JSON 추출 시도
        response_text = result.text.strip()
        
        # JSON이 다른 텍스트로 감싸져 있을 경우 처리
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0].strip()
        
        # 구조화된 응답 파싱
        recommendations = MovieList.model_validate_json(response_text)
        
        print("🎬 Movie Recommendations:\n")
        for movie in recommendations.recommendations:
            print(f"Title: {movie.title}")
            print(f"Genre: {movie.genre}")
            print(f"Year: {movie.year}")
            print(f"Rating: {movie.rating}/10")
            print(f"Reason: {movie.reason}")
            print("-" * 50)
    except json.JSONDecodeError as e:
        print(f"❌ JSON 파싱 오류: {e}")
        print(f"받은 응답: {result.text[:200]}...")
    except Exception as e:
        print(f"❌ 오류 발생: {type(e).__name__}")
        print(f"   메시지: {str(e)}")
        print(f"   받은 응답: {result.text[:200]}...")

await structured_output()

🎬 Movie Recommendations:

Title: Dune
Genre: Sci-Fi
Year: 2021
Rating: 8.2/10
Reason: A visually stunning adaptation of the classic novel, exploring themes of power and destiny.
--------------------------------------------------
Title: The Matrix Resurrections
Genre: Sci-Fi
Year: 2021
Rating: 6.8/10
Reason: A return to the iconic franchise with thought-provoking concepts of reality and identity.
--------------------------------------------------
Title: Everything Everywhere All at Once
Genre: Sci-Fi
Year: 2022
Rating: 8.7/10
Reason: A unique and imaginative exploration of multiverses, family dynamics, and self-discovery.
--------------------------------------------------


## 8. 에러 핸들링

In [9]:
async def error_handling_demo():
    try:
        # 매우 긴 입력으로 토큰 제한 테스트
        long_input = "Tell me a story. " * 10000
        result = await agent.run(long_input)
        print(result.text)
    except Exception as e:
        print(f"❌ Error occurred: {type(e).__name__}")
        print(f"   Message: {str(e)}")
        print("\n💡 Tip: Check token limits and input length")

await error_handling_demo()

Once upon a time in a small village nestled between rolling hills, there lived a clever little fox named Felix. Felix was known throughout the village for his wit and charm, but he often found himself hungry, for he had a knack for getting into trouble.

One sunny day, while wandering near the edge of the village, Felix overheard two farmers talking about a grand feast taking place in the nearby town. There would be all sorts of delicious foods: roasted meats, fresh fruits, and sweet pastries. Felix's stomach growled at the thought, and he devised a plan to get himself an invitation.

Felix approached the farmers with a bright smile. "Good afternoon, kind sirs! I overheard your plans for the feast in town. What an exciting event! I happen to be the best cook in the forest. I could cook a dish so splendid, the townsfolk would never forget it!"

The farmers, intrigued by Felix's boast, decided to take him to the town feast. They thought that having a clever fox like Felix on their side w

## 9. 성능 모니터링

Agent Framework는 OpenTelemetry를 통한 모니터링을 지원합니다.

In [10]:
import time

async def performance_monitoring():
    questions = [
        "What is AI?",
        "Explain machine learning.",
        "What is deep learning?"
    ]
    
    for question in questions:
        start_time = time.time()
        result = await agent.run(question)
        elapsed_time = time.time() - start_time
        
        print(f"Question: {question}")
        print(f"Response length: {len(result.text)} characters")
        print(f"Time taken: {elapsed_time:.2f} seconds")
        print("-" * 50)

await performance_monitoring()

Question: What is AI?
Response length: 615 characters
Time taken: 1.42 seconds
--------------------------------------------------
Question: Explain machine learning.
Response length: 1354 characters
Time taken: 2.76 seconds
--------------------------------------------------
Question: What is deep learning?
Response length: 680 characters
Time taken: 1.69 seconds
--------------------------------------------------


## 마무리

### 학습한 내용
1. ✅ Microsoft Agent Framework의 ChatAgent 기본 사용법
2. ✅ Azure OpenAI와의 통합
3. ✅ 도구(Tools) 통합 및 함수 호출
4. ✅ 스트리밍 응답 처리
5. ✅ 대화 히스토리 관리 (AgentThread)
6. ✅ 구조화된 출력 (Pydantic)
7. ✅ 에러 핸들링
8. ✅ 성능 모니터링

### 다음 단계
- `02_TeamsF.ipynb`: 멀티 에이전트 협업
- `03_Workflow.ipynb`: 워크플로우 기반 오케스트레이션